In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, ReLU

Data generation

In [ ]:
def generate_constant_data(length, value):
    return np.full((length, 1), value)

def generate_periodic_data(length, amplitude, frequency):
    t = np.linspace(0, 2 * np.pi, length)
    return amplitude * np.sin(frequency * t).reshape(-1, 1)

def generate_chaotic_data(length):
    x = np.zeros((length, 1))
    x[0] = 0.5  # 초기 조건
    for i in range(1, length):
        x[i] = 4 * x[i-1] * (1 - x[i-1])  # Logistic map
    return x
# 데이터 생성
length = 1000
num_samples_per_class = 1000

constant_data = np.array([generate_constant_data(length, value) for value in np.random.rand(num_samples_per_class)])
periodic_data = np.array([generate_periodic_data(length, amplitude=1, frequency=1) for _ in range(num_samples_per_class)])
chaotic_data = np.array([generate_chaotic_data(length) for _ in range(num_samples_per_class)])

# 라벨 생성
constant_labels = np.zeros((num_samples_per_class, 1))
periodic_labels = np.ones((num_samples_per_class, 1))
chaotic_labels = np.full((num_samples_per_class, 1), 2)

# 데이터 합치기
X_train = np.concatenate((constant_data, periodic_data, chaotic_data), axis=0)
y_train = np.concatenate((constant_labels, periodic_labels, chaotic_labels), axis=0)

# 라벨 원핫 인코딩
y_train = tf.keras.utils.to_categorical(y_train, num_classes=3)

# 데이터 셔플링
indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)
X_train = X_train[indices]
y_train = y_train[indices]

In [ ]:
# LKCNN 모델 생성 함수
def create_lkcnn(input_shape):
    model = Sequential()
    # 첫 번째 Conv1D 레이어
    model.add(Conv1D(filters=5, kernel_size=100, input_shape=input_shape))
    model.add(ReLU())
    # 두 번째 Conv1D 레이어
    model.add(Conv1D(filters=5, kernel_size=100))
    model.add(ReLU())
    # MaxPooling 레이어
    model.add(MaxPooling1D(pool_size=2))
    # Dropout 레이어
    model.add(Dropout(0.5))
    # Flatten 레이어
    model.add(Flatten())
    # 첫 번째 Dense 레이어
    model.add(Dense(100, activation='relu'))
    # 두 번째 Dense 레이어
    model.add(Dense(3, activation='softmax'))  # 세 가지 클래스를 위한 출력 레이어
    # 모델 컴파일
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:


# 모델 생성 및 훈련
input_shape = (length, 1)
model = create_lkcnn(input_shape)
model.summary()
model.fit(X_train, y_train, epochs=10, batch_size=32)

# 모델 저장
model.save('lkcnn_model.h5')

# 평가를 위한 테스트 데이터 생성 (생략 가능)
X_test_constant = np.array([generate_constant_data(length, value) for value in np.random.rand(100)])
X_test_periodic = np.array([generate_periodic_data(length, amplitude=1, frequency=1) for _ in range(100)])
X_test_chaotic = np.array([generate_chaotic_data(length) for _ in range(100)])
X_test = np.concatenate((X_test_constant, X_test_periodic, X_test_chaotic), axis=0)

y_test_constant = np.zeros((100, 1))
y_test_periodic = np.ones((100, 1))
y_test_chaotic = np.full((100, 1), 2)
y_test = np.concatenate((y_test_constant, y_test_periodic, y_test_chaotic), axis=0)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=3)

# 테스트 데이터 셔플링
indices = np.arange(X_test.shape[0])
np.random.shuffle(indices)
X_test = X_test[indices]
y_test = y_test[indices]

# 모델 평가
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy: {accuracy:.4f}")
